# Proyecto Final: Predicción de Fuga de Clientes

El operador de telecomunicaciones **Interconnect** busca reducir la cancelación de clientes (`Churn`) mediante un sistema que identifique con anticipación a quienes podrían darse de baja, para ofrecerles promociones y planes especiales.

**Objetivo:** Desarrollar y comparar modelos capaces de predecir si un cliente se dará de baja próximamente (Sí/No).

- El modelo se evaluará principalmente con AUC-ROC y con Recall como métrica adicional.

## Etapa 1: Plan de Trabajo
* Antes de tratar de hac3er algo con los datos prompongo segmentación de clientes con KMeans y luego entrenar diferentes modelos de regresión logística y de clasificación

*Escribe aquí tu plan de trabajo inicial. Aborda brevemente:*
1. *¿Cómo planeas unir los datos y qué harás con los valores nulos generados?*
2. *¿Cuál será tu variable objetivo y qué tipo de problema de Machine Learning resolverás?*
3. *¿Qué pasos de preprocesamiento (codificación categórica, fechas, etc.) consideras necesarios y sobre que variables?*
4. *¿Qué modelos planeas entrenar?*

## Etapa 2: Código de Solución

### 1. Exploración de Datos (EDA)

Descripción de los Datos

Los datos están divididos en cuatro archivos:

* `/datasets/final_provider/contract.csv`: Información del contrato (tipo de facturación, método de pago, fechas de inicio y fin).
* `/datasets/final_provider/personal.csv`: Datos demográficos del cliente.
* /datasets/final_provider/internet.csv: Información sobre los servicios de Internet contratados (fibra óptica, DSL, antivirus, etc.).
* `/datasets/final_provider/phone.csv`: Información sobre los servicios telefónicos (líneas múltiples).

*Carga de datos, análisis de distribuciones, identificación de anomalías.*

### 2. Preprocesamiento e Ingeniería de Características
*Procesar valores nulos, creación de la variable objetivo, codificación de variables categóricas (justifica tu elección de método).*
*Pista: Los modelos predictivos no entienden de fechas en formato texto. ¿Cómo puedes transformar las fechas de inicio y fin en una variable numérica útil para el modelo?*

### 3. Selección de Variables y Entrenamiento de Modelos (Baseline)
*Entrena al menos dos modelos distintos sin aplicar técnicas de balanceo de clases. Evalúa su AUC-ROC y Recall.*

### 4. Optimización y Manejo de Desbalance
*Aplica al menos una técnica para manejar el desbalance de clases (upsampling, downsampling, o ajuste de pesos) y busca los mejores hiperparámetros. Evalúa nuevamente.*

## Etapa 3: Informe de Solución
*Escribe aquí tu informe final para el equipo de negocio. Asegúrate de responder:*
1. *¿Qué modelo elegiste finalmente y por qué?*
2. *¿Cuáles fueron las métricas finales (AUC-ROC y Recall) en el conjunto de prueba?*
3. *En términos de negocio: ¿Qué significa tu valor de Recall? ¿Cómo impactaría tu modelo en la retención de clientes si el equipo de marketing lo utiliza hoy?*

# Carga y eploración


In [2]:
import warnings

import sys
import os
# Le dice python que busque liberrías ahí también
sys.path.append(os.path.join('src'))
import funciones_personales as fp

import pandas as pd

In [4]:
df_contract= pd.read_csv('datasets/final_provider/contract.csv')
df_internet= pd.read_csv('datasets/final_provider/internet.csv')
df_personal= pd.read_csv('datasets/final_provider/personal.csv')
df_phone= pd.read_csv('datasets/final_provider/phone.csv')

Contract

In [5]:
df_contract.info()
print(df_contract.head(5))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   BeginDate         7043 non-null   object 
 2   EndDate           7043 non-null   object 
 3   Type              7043 non-null   object 
 4   PaperlessBilling  7043 non-null   object 
 5   PaymentMethod     7043 non-null   object 
 6   MonthlyCharges    7043 non-null   float64
 7   TotalCharges      7043 non-null   object 
dtypes: float64(1), object(7)
memory usage: 440.3+ KB
   customerID   BeginDate              EndDate            Type  \
0  7590-VHVEG  2020-01-01                   No  Month-to-month   
1  5575-GNVDE  2017-04-01                   No        One year   
2  3668-QPYBK  2019-10-01  2019-12-01 00:00:00  Month-to-month   
3  7795-CFOCW  2016-05-01                   No        One year   
4  9237-HQITU  2019-09-01  2019-11-01 00

In [18]:
print(df_contract['Type'].unique())
df_contract.describe(include='all')

['Month-to-month' 'One year' 'Two year']


,customerID,BeginDate,EndDate,Type,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
count,7043,7043,7043,7043,7043,7043,7043.000000,7043
unique,7043,77,5,3,2,4,NaN,6531
top,3186-AJIEK,2014-02-01,No,Month-to-month,Yes,Electronic check,NaN,
freq,1,366,5174,3875,4171,2365,NaN,11
mean,NaN,NaN,NaN,NaN,NaN,NaN,64.761692,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,30.090047,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,18.250000,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,35.500000,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,70.350000,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,89.850000,NaN


7,043 datos con diferentes id sin valores nulos

Las fechas son tipo object.
* BeginDate es formato YYYY-DD-MM
* EndDate es formato YYYY-DD-MM hh:mm:ss 
    * EndDate será importante para determinar los que hicieron churn
Teniendo 77 fechas diferentes podemos inferir información de 77 días
**requiere análisis posterior**

* ya que los valores respetidos para
```
EndDate                5
Type                   3
PaperlessBilling       2
PaymentMethod          4
```
son pocos, no sé su sustiturlos con numeros del uno al 5 o dejarlos como están

* Total_Charges No estoy seguro de que sirva para el modelo ya que son consecuencia directa de el tipo y el monthly charges, además son object (no Float)

internet

In [6]:
df_internet.info()
print(df_internet.head(5))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5517 entries, 0 to 5516
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   customerID        5517 non-null   object
 1   InternetService   5517 non-null   object
 2   OnlineSecurity    5517 non-null   object
 3   OnlineBackup      5517 non-null   object
 4   DeviceProtection  5517 non-null   object
 5   TechSupport       5517 non-null   object
 6   StreamingTV       5517 non-null   object
 7   StreamingMovies   5517 non-null   object
dtypes: object(8)
memory usage: 344.9+ KB
   customerID InternetService OnlineSecurity OnlineBackup DeviceProtection  \
0  7590-VHVEG             DSL             No          Yes               No   
1  5575-GNVDE             DSL            Yes           No              Yes   
2  3668-QPYBK             DSL            Yes          Yes               No   
3  7795-CFOCW             DSL            Yes           No              Yes   
4 

In [15]:
print(df_internet.nunique())

customerID          5517
InternetService        2
OnlineSecurity         2
OnlineBackup           2
DeviceProtection       2
TechSupport            2
StreamingTV            2
StreamingMovies        2
dtype: int64


5,517 datos sin valores nulos
* 5,517 clientes cuentan con internet: el resto no.
    * comprobar que todos los id de `df_internet` existan en `df_contract`
* Todas las columnas excepto `'customer_ID` e internet service son "yer or no" (1 o 0), ¿cambiarlos a 0 y 1 ayudará al modelo?

Personal

In [7]:
df_personal.info()
print(df_personal.head(5))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   customerID     7043 non-null   object
 1   gender         7043 non-null   object
 2   SeniorCitizen  7043 non-null   int64 
 3   Partner        7043 non-null   object
 4   Dependents     7043 non-null   object
dtypes: int64(1), object(4)
memory usage: 275.2+ KB
   customerID  gender  SeniorCitizen Partner Dependents
0  7590-VHVEG  Female              0     Yes         No
1  5575-GNVDE    Male              0      No         No
2  3668-QPYBK    Male              0      No         No
3  7795-CFOCW    Male              0      No         No
4  9237-HQITU  Female              0      No         No


In [16]:
print(df_personal.nunique())


customerID       7043
gender              2
SeniorCitizen       2
Partner             2
Dependents          2
dtype: int64


7,043 datos sin valores nulos
* Verificar que todos los ID coincidan con los de df_contract
* De nuevo todas las respuestas de las columnas son si ó no (0's y 1's)

phone

In [8]:
df_phone.info()
print(df_phone.head(5))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6361 entries, 0 to 6360
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   customerID     6361 non-null   object
 1   MultipleLines  6361 non-null   object
dtypes: object(2)
memory usage: 99.5+ KB
   customerID MultipleLines
0  5575-GNVDE            No
1  3668-QPYBK            No
2  9237-HQITU            No
3  9305-CDSKC           Yes
4  1452-KIOVK           Yes


In [17]:
print(df_phone.nunique())

customerID       6361
MultipleLines       2
dtype: int64


6,361 datos sin valores nulos
* Verificar qie todos los id existan en contracts
* ¿cambiar a 1 y 0?